from libraries import *
from parameters import *
from Utilities import *

In [ ]:
os.getcwd()
os.chdir(projectDir)

In [ ]:
%load_ext rpy2.ipython

In [ ]:
conf_mt_prefix = 'MT-' if par_species == 'human' else 'mt-'

In [ ]:
adata = sc.read("./Data/anndata_guide.h5ad")

In [ ]:
adataGuides = adata[adata.obs["guide_num"] == 1,:]
availTFs = [x for x in adataGuides.obs["guide_id"].unique() if x in list(adataGuides.var["gene_name"].unique())]


In [ ]:
del adataGuides

In [ ]:
adata.var

In [ ]:
adataNeuro = sc.read("./outputs/anndata/adataNeuro.h5ad")

In [ ]:
adataNeuro.var["gene_name"] = list(adataNeuro.var.index)

In [ ]:
adataNeuro.obs["guide_id"] = "None"
adataNeuro.obs["full_guide_id"] = "None"
adataNeuro.obs["guide_num"] = 100

In [ ]:
adata.obs["Sample_type"] = "KO_Screen"
adata.obs['n_umis']  = adata.X.sum(1)
adata.obs['n_genes'] = (adata.X != 0).sum(1).A1
adata.obs['log10_n_umis'] = np.log10(adata.X.sum(1))
adata.obs['log10_n_genes'] = np.log10((adata.X != 0).sum(1).A1)

mt_gene_mask = adata.var_names.str.startswith(conf_mt_prefix)
assert mt_gene_mask.sum() > 0, 'Wrong mt prefix'
adata.obs['mt_frac'] = adata.X[:, mt_gene_mask].sum(1).A1 / adata.obs['n_umis']


In [ ]:
counts, bins = np.histogram(adata.obs["guide_num"], bins=100)
plt.stairs(counts, bins)
plt.xlabel('Number of guides')
plt.ylabel('Number of cells')

In [ ]:
adata = adata[(adata.obs["guide_num"] == 1) | (adata.obs["guide_num"] == 2),:]

In [ ]:
adata.obs["demux_type"] = "Single_guide"
adata.obs.loc[adata.obs["guide_num"] == 2,"demux_type"] = "Double_guide"


In [ ]:
from matplotlib.pyplot import rc_context

with rc_context({'figure.figsize': (4, 4)}):
    sc.pl.violin(adata, ['mt_frac'], 
                stripplot=False, inner='box', groupby='demux_type')

In [ ]:
from matplotlib.pyplot import rc_context

with rc_context({'figure.figsize': (4, 4)}):
    sc.pl.violin(adata, ['log10_n_umis'], 
                stripplot=False, inner='box', groupby='demux_type')

In [ ]:
from matplotlib.pyplot import rc_context

with rc_context({'figure.figsize': (4, 4)}):
    sc.pl.violin(adata, ['n_genes'], 
                stripplot=False, inner='box', groupby='demux_type')

In [ ]:
adata.obs["guide_num"].value_counts()

In [ ]:
adata = adata[adata.obs["mt_frac"] < 0.2,:]
adata = adata[adata.obs["n_genes"] < 5000,:]
adata = adata[adata.obs["n_genes"] > 500,:]

In [ ]:
adata.obs["guide_num"].value_counts()

In [ ]:
#guideMat = adata.obs

In [ ]:
# %%R -i guideMat

# guideMat$guide_id = as.character(guideMat$guide_id)
# guideMat$full_guide_id = as.character(guideMat$full_guide_id)

# kk = data.frame(table(guideMat$guide_id))
# kk = kk[order(kk$Freq),]
# kk

In [ ]:
sc.pp.filter_genes(adata, min_cells = 0, inplace=True)

In [ ]:
geneSignatures = pd.DataFrame(pd.read_csv("./TextFiles/Human_NE_signatures.csv"))

In [ ]:
# List of Ensembl IDs you want to search for
lowNEGenes = list(geneSignatures["Low in NE"]) # Add more IDs as needed

# Create a regular expression pattern that matches any of the IDs
pattern = r'\b(?:' + '|'.join(lowNEGenes) + r')\b'


# Search for the rows in adata.var where the 'gene_id' contains any of the given Ensembl IDs
filtered_adata = adata.var[adata.var['gene_name'].str.contains(pattern)]
filtered_adata = filtered_adata.sort_values(by='n_cells', ascending=True)



In [ ]:
selGenes1 = list(filtered_adata["gene_id"].unique())

In [ ]:
plt.figure(figsize=(10, 6))  # Adjust the size as needed

plt.yscale('log', base=10)

# Create the bar plot
plt.bar(filtered_adata['gene_name'], filtered_adata['n_cells'])

plt.xticks(rotation=90, fontsize=10)

# Add labels and title
plt.xlabel('Gene')
plt.ylabel('Number of cells')
plt.title('Adenocarcinoma Genes')

# Show the plot
plt.show()

In [ ]:
# List of Ensembl IDs you want to search for
highNEGenes = list(geneSignatures["High in NE"]) # Add more IDs as needed

highNEGenes = [s for s in highNEGenes if not pd.isna(s)]


# Create a regular expression pattern that matches any of the IDs
pattern = r'\b(?:' + '|'.join(highNEGenes) + r')\b'


# Search for the rows in adata.var where the 'gene_id' contains any of the given Ensembl IDs
filtered_adata = adata.var[adata.var['gene_name'].str.contains(pattern)]
filtered_adata = filtered_adata.sort_values(by='n_cells', ascending=True)



In [ ]:
selGenes2 = list(filtered_adata["gene_id"].unique())

In [ ]:
plt.figure(figsize=(10, 6))  # Adjust the size as needed

plt.yscale('log', base=10)

# Create the bar plot
plt.bar(filtered_adata['gene_name'], filtered_adata['n_cells'])

plt.xticks(rotation=90, fontsize=10)

# Add labels and title
plt.xlabel('Gene')
plt.ylabel('Number of cells')
plt.title('NEPC Genes')

# Show the plot
plt.show()

In [ ]:
nepcSignatures = pd.DataFrame(pd.read_csv("./TextFiles/NEPC_adenocarcinoma_markerGenes.csv", index_col=0))

In [ ]:
for i in  nepcSignatures.columns:
    print(i)
    myGeneList = [x for x in nepcSignatures.loc[:,i] if x != ' ']
    myGeneList = [x for x in myGeneList if x !="nan"]
    myGeneList = [x for x in myGeneList if not isinstance(x, float)]

    myGeneList = [x.replace(".","-") for x in myGeneList]
    
    myGeneList = [x for x in myGeneList if x in adataALL.var_names]
    
#     sc.pl.heatmap(adata_ref, myGeneList, groupby='leiden', dendrogram=True)
#     sc.pl.dotplot(adata_ref, myGeneList, groupby='leiden', dendrogram=True)
    
    if(sum(pd.Series(myGeneList).isin(adataALL.var_names)) > 1):
        sc.tl.score_genes(adata=adataALL, gene_list=myGeneList, score_name=i)
        sc.pl.umap(adataALL, color=i, size=0.5, color_map="coolwarm")
        #f, ax = plt.subplots(figsize=(12, 4))
        #sc.pl.violin(adata_ref, i, groupby='leiden', ax=ax)
        


In [ ]:
# List of Ensembl IDs you want to search for
NEPCGenes = list(nepcSignatures["NEPC"]) # Add more IDs as needed

NEPCGenes = [s for s in NEPCGenes if not pd.isna(s)]


# Create a regular expression pattern that matches any of the IDs
pattern = r'\b(?:' + '|'.join(NEPCGenes) + r')\b'


# Search for the rows in adata.var where the 'gene_id' contains any of the given Ensembl IDs
filtered_adata = adata.var[adata.var['gene_name'].str.contains(pattern)]
filtered_adata = filtered_adata.sort_values(by='n_cells', ascending=True)
filtered_adata

In [ ]:
plt.figure(figsize=(10, 6))  # Adjust the size as needed

plt.yscale('log', base=10)

# Create the bar plot
plt.bar(filtered_adata['gene_name'], filtered_adata['n_cells'])

plt.xticks(rotation=90, fontsize=10)

# Add labels and title
plt.xlabel('Gene')
plt.ylabel('Number of cells')
plt.title('NEPC Genes')

# Show the plot
plt.show()

In [ ]:
selGenes3 = list(filtered_adata["gene_id"].unique())

In [ ]:
# List of Ensembl IDs you want to search for
adenocarcinomaGenes = list(nepcSignatures["adenocarcinoma"]) # Add more IDs as needed

adenocarcinomaGenes = [s for s in adenocarcinomaGenes if not pd.isna(s)]


# Create a regular expression pattern that matches any of the IDs
pattern = r'\b(?:' + '|'.join(adenocarcinomaGenes) + r')\b'


# Search for the rows in adata.var where the 'gene_id' contains any of the given Ensembl IDs
filtered_adata = adata.var[adata.var['gene_name'].str.contains(pattern)]
filtered_adata = filtered_adata.sort_values(by='n_cells', ascending=True)
filtered_adata

In [ ]:
selGenes4 = list(filtered_adata["gene_id"].unique())

In [ ]:
plt.figure(figsize=(10, 6))  # Adjust the size as needed

plt.yscale('log', base=10)

# Create the bar plot
plt.bar(filtered_adata['gene_name'], filtered_adata['n_cells'])

plt.xticks(rotation=90, fontsize=10)

# Add labels and title
plt.xlabel('Gene')
plt.ylabel('Number of cells')
plt.title('NEPC Genes')

# Show the plot
plt.show()

In [ ]:
adataNeuro.var["gene_id"] = adataNeuro.var["gene_ids"]

In [ ]:
#sc.pp.filter_genes(adata, min_cells = 10, inplace=True)

In [ ]:
def make_gene_names_unique(df, name_col='gene_name', score_col='n_cells'):
    from collections import defaultdict

    used_names = set()
    counts = defaultdict(int)
    result = []

    # Group by gene name and find the index with the max score
    max_indices = set(df.groupby(name_col)[score_col].idxmax().values)

    for i, row in df.iterrows():
        base_name = row[name_col]

        if i in max_indices:
            new_name = base_name
        else:
            counts[base_name] += 1
            new_name = f"{base_name}_REP{counts[base_name]}"

        # Ensure global uniqueness
        while new_name in used_names:
            counts[base_name] += 1
            new_name = f"{base_name}_REP{counts[base_name]}"

        used_names.add(new_name)
        result.append(new_name)

    return result

adata.var["gene_name_unique"] = make_gene_names_unique(adata.var)

In [ ]:
adata.var

In [ ]:
adataNeuro.var

In [ ]:
adata.var.set_index("gene_id", inplace=True) 

In [ ]:
adataNeuro.var.set_index("gene_id", inplace=True) 

In [ ]:
adata.var["gene_ids"] = list(adata.var.index)

In [ ]:
adata.var

In [ ]:
selGenes5 = list(adata.var.loc[[x in availTFs for x in list(adata.var["gene_name_unique"].unique())],"gene_ids"].unique())

In [ ]:
adataALL_tmp = sc.AnnData.concatenate(adata, adataNeuro, join="inner", batch_key="sample_name")
sc.pp.filter_genes(adataALL_tmp, min_cells = 1000, inplace=True)
selGenes6 = list(adataALL_tmp.var["gene_ids"])

In [ ]:
adataNeuro_tmp =adataNeuro.copy()
sc.pp.normalize_total(adataNeuro_tmp, target_sum=par_preprocessing_target_sum)
sc.pp.log1p(adataNeuro_tmp)
sc.pp.scale(adataNeuro_tmp, max_value=10)
sc.tl.rank_genes_groups(adataNeuro_tmp, groupby="Sample_type", n_genes=1000, method="t-test_overestim_var")
markerGenes = pd.DataFrame(adataNeuro_tmp.uns['rank_genes_groups']['names'])
selGenes7 = list(markerGenes["adenocarcinoma"])

In [ ]:
selGenes = list(set(selGenes1 + selGenes2 + selGenes3 + selGenes4 + selGenes5 + selGenes6 + selGenes7))
len(selGenes)

In [ ]:
adataALL = sc.AnnData.concatenate(adata, adataNeuro, 
                                  join="inner", 
                                  batch_key="sample_name")


In [ ]:
del adata
del adataALL_tmp
del adataNeuro_tmp
del adataNeuro

In [ ]:
selGenes = [x  for x in selGenes if x in adataALL.var["gene_ids"]]

In [ ]:
len(selGenes)

In [ ]:
adataALL = adataALL[:,selGenes]

In [ ]:
adataALL.var['gene_name-1'] = adataALL.var['gene_name-1'].astype(str)

In [ ]:
adataALL.var.set_index("gene_name-1", inplace=True) 

In [ ]:
adataALL

In [ ]:
sc.pp.filter_genes(adataALL, min_cells = 15, inplace=True)

In [ ]:
adataALL

In [ ]:
adataALL.layers['counts'] = adataALL.X.copy()

sc.pp.normalize_total(adataALL, target_sum=par_preprocessing_target_sum)
sc.pp.log1p(adataALL)
adataALL.raw = adataALL

In [ ]:
sc.pp.scale(adataALL, max_value=10)

In [ ]:
gene_list_url = 'https://raw.githubusercontent.com/theislab/scanpy_usage/master/180209_cell_cycle/data/regev_lab_cell_cycle_genes.txt'

cell_cycle_genes = [str(x.strip(), 'utf-8').upper() for x in urlopen(gene_list_url)] 
s_genes = cell_cycle_genes[:43]
g2m_genes = cell_cycle_genes[43:]


In [ ]:
sc.tl.score_genes_cell_cycle(adataALL, s_genes=s_genes, g2m_genes=g2m_genes)

In [ ]:
sc.pp.regress_out(adataALL, ['S_score','G2M_score','log10_n_umis',
                              'mt_frac', 'n_genes'], n_jobs=60)

In [ ]:
sc.pp.combat(adataALL,"sample_name")

In [ ]:
del adataALL.var['highly_variable']

In [ ]:
sc.pp.highly_variable_genes(adataALL, n_top_genes=5000)


In [ ]:
sc.pp.pca(adataALL, n_comps=50, svd_solver='arpack')


In [ ]:
sc.pp.neighbors(adataALL, n_neighbors=5, n_pcs=50)


In [ ]:
sc.tl.umap(adataALL)

In [ ]:
adataALL.obs

In [ ]:
sc.pl.umap(adataALL, color="Sample_type", size=0.5,
           color_map="coolwarm", vmax=2.0, palette=["yellow", "red", "blue"])


In [ ]:
#adata_ref.write("./Data/adata_ref.h5ad")

In [ ]:
ifnSignatures = pd.DataFrame(pd.read_csv("./TextFiles/ifn_signatures_oana.csv", index_col=None))

In [ ]:
for i in  ifnSignatures.columns:
    print(i)
    myGeneList = [x for x in ifnSignatures.loc[:,i] if x != ' ']
    myGeneList = [x for x in myGeneList if x !="nan"]
    myGeneList = [x for x in myGeneList if not isinstance(x, float)]

    myGeneList = [x.replace(".","-") for x in myGeneList]
    
    myGeneList = [x for x in myGeneList if x in adataALL.var_names]
    
#     sc.pl.heatmap(adata_ref, myGeneList, groupby='leiden', dendrogram=True)
#     sc.pl.dotplot(adata_ref, myGeneList, groupby='leiden', dendrogram=True)
    
    if(sum(pd.Series(myGeneList).isin(adataALL.var_names)) > 1):
        sc.tl.score_genes(adata=adataALL, gene_list=myGeneList, score_name=i)
        sc.pl.umap(adataALL, color=i, size=0.5, color_map="coolwarm", vmax=0.3)
        f, ax = plt.subplots(figsize=(12, 4))
        sc.pl.violin(adataALL, i, groupby='leiden', ax=ax)
  

In [ ]:
adataALL.obs['leiden'].value_counts()

In [ ]:
adata_4_1_0 = adataALL[adataALL.obs['leiden'] == '4_1_0',]
tmp = pd.DataFrame(adata_4_1_0.obs['guide_id'].value_counts())
selKos = list(tmp[0:85].index)
adata_4_1_0 = adata_4_1_0[[x in selKos for x in adata_4_1_0.obs['guide_id']],]
violinPlotObsValue("IFNG_signature", "guide_id", adata_4_1_0)

In [ ]:
tmp[0:45]

In [ ]:
adata_4_1_1 = adataALL[adataALL.obs['leiden'] == '4_1_1',]
tmp = pd.DataFrame(adata_4_1_1.obs['guide_id'].value_counts())
tmp[0:15]

In [ ]:
geneSignatures = pd.DataFrame(pd.read_csv("./TextFiles/Human_NE_signatures.csv"))

In [ ]:
for i in  geneSignatures.columns:
    print(i)
    myGeneList = [x for x in geneSignatures.loc[:,i] if x != ' ']
    myGeneList = [x for x in myGeneList if x !="nan"]
    myGeneList = [x for x in myGeneList if not isinstance(x, float)]

    myGeneList = [x.replace(".","-") for x in myGeneList]
    
    myGeneList = [x for x in myGeneList if x in adataALL.var_names]
    
#     sc.pl.heatmap(adata_ref, myGeneList, groupby='leiden', dendrogram=True)
#     sc.pl.dotplot(adata_ref, myGeneList, groupby='leiden', dendrogram=True)
    
    if(sum(pd.Series(myGeneList).isin(adataALL.var_names)) > 1):
        sc.tl.score_genes(adata=adataALL, gene_list=myGeneList, score_name=i)
        sc.pl.umap(adataALL, color=i, size=0.5, color_map="coolwarm", vmax=0.5)
        #f, ax = plt.subplots(figsize=(12, 4))
        #sc.pl.violin(adata_ref, i, groupby='leiden', ax=ax)
        


In [ ]:
for i in  nepcSignatures.columns:
    print(i)
    myGeneList = [x for x in nepcSignatures.loc[:,i] if x != ' ']
    myGeneList = [x for x in myGeneList if x !="nan"]
    myGeneList = [x for x in myGeneList if not isinstance(x, float)]

    myGeneList = [x.replace(".","-") for x in myGeneList]
    
    myGeneList = [x for x in myGeneList if x in adataALL.var_names]
    
#     sc.pl.heatmap(adata_ref, myGeneList, groupby='leiden', dendrogram=True)
#     sc.pl.dotplot(adata_ref, myGeneList, groupby='leiden', dendrogram=True)
    
    if(sum(pd.Series(myGeneList).isin(adataALL.var_names)) > 1):
        sc.tl.score_genes(adata=adataALL, gene_list=myGeneList, score_name=i)
        sc.pl.umap(adataALL, color=i, size=0.5, color_map="coolwarm")
        #f, ax = plt.subplots(figsize=(12, 4))
        #sc.pl.violin(adata_ref, i, groupby='leiden', ax=ax)
        

In [ ]:
for elem in ['ASCL1']:
    sc.pl.umap(adataALL, color=elem, size=0.5,
               color_map="coolwarm", vmax=2.0)


In [ ]:
sc.tl.leiden(adataALL, resolution=0.6)


In [ ]:
adataALL = sc.read("./Data/adataALL.h5ad")

In [ ]:
f, ax = plt.subplots(figsize=(4, 4))
sc.pl.umap(adataALL, color='leiden', 
           legend_fontoutline=3, legend_fontsize=14, 
           legend_fontweight='normal', title='Clusters', ax=ax, show=False, size=0.5, palette="tab20"
);

In [ ]:
adataALL.obs['leiden'].unique()

In [ ]:
adataALL.obs['leiden'].value_counts()

In [ ]:
adataALLTC3 = adataALL[adataALL.obs['leiden'] == '3',:]

sc.tl.leiden(adataALLTC3, resolution=0.1, key_added='TC_3_reclustered')

sc.pl.umap(adataALLTC3, color='TC_3_reclustered')

adataALLTC3.obs.loc[adataALLTC3.obs["TC_3_reclustered"] == '2',"TC_3_reclustered"]='0'

In [ ]:
adataALLTC3.obs.loc[adataALLTC3.obs["TC_3_reclustered"] == '2',"TC_3_reclustered"]='0'

In [ ]:
adataALLTC3.obs['leiden'] = ['3_'+x for x in adataALLTC3.obs["TC_3_reclustered"]]

In [ ]:
adataALL.obs['leiden'] = adataALL.obs['leiden'].astype(str)
adataALL.obs.loc[adataALL.obs['leiden'] == '3','leiden'] = adataALLTC3.obs['leiden']

In [ ]:
adataALLTC4 = adataALL[adataALL.obs['leiden'] == '4',:]

sc.tl.leiden(adataALLTC4, resolution=0.1, key_added='TC_4_reclustered')

sc.pl.umap(adataALLTC4, color='TC_4_reclustered')

In [ ]:
adataALLTC4.obs['leiden'] = ['4_'+x for x in adataALLTC4.obs["TC_4_reclustered"]]

In [ ]:
adataALL.obs['leiden'] = adataALL.obs['leiden'].astype(str)
adataALL.obs.loc[adataALL.obs['leiden'] == '4','leiden'] = adataALLTC4.obs['leiden']

In [ ]:
f, ax = plt.subplots(figsize=(4, 4))
adataALL.obs['leiden'] = adataALL.obs['leiden'].astype(str)

sc.pl.umap(adataALL, color='leiden', 
           legend_fontoutline=3, legend_fontsize=14, 
           legend_fontweight='normal', title='Clusters', ax=ax, show=False, size=0.5, palette="tab20"
);

In [ ]:
for elem in adataALL.obs['leiden'].unique():
    adataALL.obs["Temp"] = "Others"
    adataALL.obs.loc[adataALL.obs['leiden'] == elem,"Temp"] = elem
    f, ax = plt.subplots(figsize=(4, 4))

    sc.pl.umap(adataALL, color='Temp', 
           legend_fontoutline=3, legend_fontsize=14, 
           legend_fontweight='normal', title='Clusters', ax=ax, show=False, size=0.5, palette=["red","lightgrey"]
    );

In [ ]:
sc.tl.dendrogram(adataALL, groupby="leiden")

In [ ]:
sc.pl.dendrogram(adataALL, groupby="leiden")

In [ ]:
adataALL.obs.loc[adataALL.obs["leiden"] == '12',"leiden"]='3_1_0'
adataALL.obs.loc[adataALL.obs["leiden"] == '10',"leiden"]='3_1_1'

In [ ]:
adataALL.obs.loc[adataALL.obs["leiden"] == '16',"leiden"]='0'
adataALL.obs.loc[adataALL.obs["leiden"] == '15',"leiden"]='0'
adataALL.obs.loc[adataALL.obs["leiden"] == '14',"leiden"]='1'
adataALL.obs.loc[adataALL.obs["leiden"] == '13',"leiden"]='1'

In [ ]:
adataALL.obs.loc[adataALL.obs["leiden"] == '11',"leiden"]='1'

In [ ]:
adataALLTC3 = adataALL[adataALL.obs['leiden'] == '3_1',:]

sc.tl.leiden(adataALLTC3, resolution=0.1, key_added='TC_3_reclustered')


adataALLTC3.obs.loc[adataALLTC3.obs["TC_3_reclustered"] == '2',"TC_3_reclustered"]='1'

sc.pl.umap(adataALLTC3, color='TC_3_reclustered')


In [ ]:
adataALLTC3.obs['leiden'] = ['3_1_'+x for x in adataALLTC3.obs["TC_3_reclustered"]]

In [ ]:
adataALL.obs['leiden'] = adataALL.obs['leiden'].astype(str)
adataALL.obs.loc[adataALL.obs['leiden'] == '3_1','leiden'] = adataALLTC3.obs['leiden']

In [ ]:
adataALLTC4 = adataALL[adataALL.obs['leiden'] == '4_1',:]

sc.tl.leiden(adataALLTC4, resolution=0.1, key_added='TC_4_reclustered')


adataALLTC4.obs.loc[adataALLTC4.obs["TC_4_reclustered"] == '2',"TC_4_reclustered"]='0'
adataALLTC4.obs.loc[adataALLTC4.obs["TC_4_reclustered"] == '3',"TC_4_reclustered"]='0'
adataALLTC4.obs.loc[adataALLTC4.obs["TC_4_reclustered"] == '4',"TC_4_reclustered"]='0'

adataALLTC4.obs['TC_4_reclustered'] = adataALLTC4.obs['TC_4_reclustered'].astype(str)

sc.pl.umap(adataALLTC4, color='TC_4_reclustered')


In [ ]:
adataALLTC4.obs['leiden'] = ['4_1_'+x for x in adataALLTC4.obs["TC_4_reclustered"]]

In [ ]:
adataALL.obs['leiden'] = adataALL.obs['leiden'].astype(str)
adataALL.obs.loc[adataALL.obs['leiden'] == '4_1','leiden'] = adataALLTC4.obs['leiden']

In [ ]:
adataALLTC6 = adataALL[adataALL.obs['leiden'] == '6',:]

sc.tl.leiden(adataALLTC6, resolution=0.1, key_added='TC_6_reclustered')

adataALLTC6.obs.loc[adataALLTC6.obs["TC_6_reclustered"] == '2',"TC_6_reclustered"]='0'

sc.pl.umap(adataALLTC6, color='TC_6_reclustered')


In [ ]:
adataALLTC6.obs['leiden'] = ['6_'+x for x in adataALLTC6.obs["TC_6_reclustered"]]

In [ ]:
adataALL.obs['leiden'] = adataALL.obs['leiden'].astype(str)
adataALL.obs.loc[adataALL.obs['leiden'] == '6','leiden'] = adataALLTC6.obs['leiden']

In [ ]:
f, ax = plt.subplots(figsize=(4, 4))
adataALL.obs['leiden'] = adataALL.obs['leiden'].astype(str)

sc.pl.umap(adataALL, color='leiden', 
           legend_fontoutline=3, legend_fontsize=14, 
           legend_fontweight='normal', title='Clusters', ax=ax, show=False, size=0.5, palette="tab20"
);

In [ ]:
tmp =pd.DataFrame(adataALL.obs['leiden'].value_counts())
tmp["cluster"] = list(tmp.index)

In [ ]:

plt.figure(figsize=(8, 4))  # Adjust the size as needed

plt.yscale('log', base=10)

# Create the bar plot
plt.bar(tmp['cluster'], tmp['leiden'])

plt.xticks(rotation=90, fontsize=10)

# Add labels and title
plt.xlabel('Cluster')
plt.ylabel('Number of cells')
plt.title('Cluster size')

# Show the plot
plt.show()

In [ ]:
f, ax = plt.subplots(figsize=(4, 4))
sc.pl.umap(adataALL, color='leiden', 
           legend_fontoutline=3, legend_fontsize=7, legend_loc="on data",
           legend_fontweight='normal', title='Clusters', ax=ax, show=False, size=0.5,
           palette='tab20'
);

In [ ]:
kk = pd.DataFrame(pd.crosstab(adataALL.obs[ "leiden"], adataALL.obs["Sample_type"]))
kk = kk / kk.sum(axis=0)
kk["leiden"] = list(kk.index)

In [ ]:
%%R -i kk -w 7 -h 6 -u in

library('ggplot2')
library(RColorBrewer)
library(reshape2)

kkMelted = melt(kk, id.vars="leiden")
kkMelted$leiden = factor(kkMelted$leiden )
mycolors <- colorRampPalette(brewer.pal(8, "Paired"))(15)

mycolors

ggplot(kkMelted, aes(x=as.factor(leiden),y=value, fill=leiden )) + 
  xlab("Clusters")+
  ylab("Cell population percentage")+
  facet_wrap(~variable, ncol=1)+
  geom_bar(stat = "identity" ) +
  theme(legend.position="none")+theme_minimal()+
  scale_fill_manual(values=mycolors)


In [ ]:
for elem in ["CDKN1A", "CDKN2A", "EPCAM", "CDH1", "ACSL1", "AR"]:
    sc.pl.umap(adataALL, color=elem, size=0.5,
               color_map="viridis", vmax=2)


In [ ]:
adataALL.obs['leiden'].value_counts()

In [ ]:
violinPlotObsValue("Low in NE", "leiden", adataALL)

In [ ]:
violinPlotObsValue("High in NE", "leiden", adataALL)

In [ ]:
violinPlotObsValue("NEPC", "leiden", adataALL)

In [ ]:
violinPlotObsValue("adenocarcinoma", "leiden", adataALL)

In [ ]:
def violinPlotObsValue(columnName, groupName, adata, testedPairs=None):
    
    # Extract the gene expression and group
    df = pd.DataFrame({
        'expression': adata.obs[columnName],  # or .X.flatten() if dense
        'group': adata.obs[groupName].values
    })

    group_medians = df.groupby("group")["expression"].median()

    # 2. Sort the groups based on median
    sorted_groups = group_medians.sort_values().index.tolist()


    # Plot
    plt.figure(figsize=(16,6))
    ax = sns.boxplot(x="group", y="expression", data=df, order=sorted_groups)
    ax.set_ylabel(columnName)
    ax.set_xticklabels(ax.get_xticklabels(),rotation=30, size=5 )



    # annotator = Annotator(ax, testedPairs, data=df, x="group", y="expression")
    # annotator.configure(test='Mann-Whitney-gt', text_format='star', loc='outside')
    # annotator.apply_and_annotate()

    plt.show()
def violinPlotGeneWithSignificance(geneName, groupName, adata, testedPairs):
    
    # Extract the gene expression and group
    df = pd.DataFrame({
        'expression': adata[:, geneName].X.toarray().flatten(),  # or .X.flatten() if dense
        'group': adata.obs[groupName].values
    })

    # Plot
    plt.figure(figsize=(8,6))
    ax = sns.boxplot(x="group", y="expression", data=df)
    ax.set_ylabel("ASCL1 expression")

    # Add significance test


    annotator = Annotator(ax, testedPairs, data=df, x="group", y="expression")
    annotator.configure(test='Mann-Whitney', text_format='star', loc='outside')
    annotator.apply_and_annotate()

    plt.show()



In [ ]:
adataSub = adataALL[[x in ["NTC", "TEAD1", "TEAD2", "TEAD3", "TEAD4"] for x in adataALL.obs["guide_id"]],:]

pairs = [
        ("NTC", "TEAD1"),
        ("NTC", "TEAD2"),
        ("NTC", "TEAD3"),
        ("NTC", "TEAD4")
        # add more group pairs if needed
    ]

violinPlotGeneWithSignificance(geneName='ASCL1', groupName= "guide_id", adata=adataSub, testedPairs=pairs)

In [ ]:
#sc.tl.ingest(adata_unpert, adata_ref, obs="leiden")


In [ ]:
#adataALL.write("./Data/adataALL.h5ad")

In [ ]:
adataALL.obs["Sample_type"] = adataALL.obs["Sample_type"].cat.set_categories(['KO_Screen_NTC',"KO_Screen_KO" ,"NEPC","adenocarcinoma"])
adataALL.obs.loc[adataALL.obs["guide_id"] == "NTC","Sample_type"] = "KO_Screen_NTC"
adataALL.obs.loc[(adataALL.obs["guide_id"] != "NTC") & (adataALL.obs["guide_id"] != "None"),"Sample_type"] = "KO_Screen_KO"

In [ ]:
sc.pl.umap(adataALL, color="Sample_type", size=3,
           color_map="coolwarm", vmax=2.0, palette=["green", "lightgrey", "blue", "magenta"])


In [ ]:
adataALL.uns['log1p']['base']=None
sc.tl.rank_genes_groups(adataALL, groupby="leiden", n_genes=2000, method="t-test_overestim_var")
sc.tl.dendrogram(adataALL, groupby='leiden')


In [ ]:
sc.pl.rank_genes_groups_matrixplot(adataALL, n_genes=10, standard_scale='var', cmap='Blues')

In [ ]:
markerGenes = pd.DataFrame(adataALL.uns['rank_genes_groups']['names'])
markerGenes = markerGenes.iloc[0:40,:]
markerGenes.to_csv("./TextFiles/Leiden_markerGenes.csv")

In [ ]:
for i in  markerGenes.columns:
    myGeneList = [x for x in markerGenes.loc[:,i] if x != ' ']
    myGeneList = [x for x in markerGenes.loc[:,i] if x !="nan"]
    myGeneList = [x for x in markerGenes.loc[:,i] if not isinstance(x, float)]

    myGeneList = [x.replace(".","-") for x in myGeneList]

    print(myGeneList)
    if(sum(pd.Series(myGeneList).isin(adataALL.var_names)) > 1):
        sc.tl.score_genes(adata=adataALL, gene_list=myGeneList, score_name=i)
        sc.pl.umap(adataALL, color=i, size=10, color_map="coolwarm")
        f, ax = plt.subplots(figsize=(12, 4))
        sc.pl.violin(adataALL, i, groupby='leiden', ax=ax)

In [ ]:
for elem in list(adata_ref.var.loc[[x in selGenes1 for x in adata_ref.var["gene_ids"]],:].index):
    sc.pl.umap(adata_ref, color=elem, size=0.5,
               color_map="coolwarm", vmax=2.0)


In [ ]:
adata_ref.var.loc[adata_ref.var["gene_name-0"] == "AR",:]

In [ ]:
adataALL_new.var.loc[adataALL_new.var["gene_name-0"] == "AR",:]

In [ ]:
adata_unpert.var.loc[adata_unpert.var["gene_name-0"] == "AR",:]

In [ ]:
sc.pl.dotplot(adataALL, ["CDKN1A", "CDKN2A", "EPCAM", "CDH1", "ASCL1", "AR"],
              groupby='leiden', dendrogram=True, standard_scale='var' )

In [ ]:
for elem in ["CDKN1A", "CDKN2A", "EPCAM", "CDH1", "ACSL1", "AR"]:
    sc.pl.umap(adata_ref, color=elem, size=0.5,
               color_map="viridis", vmax=1)
    



In [ ]:
for elem in [ "AR"]:
    sc.pl.umap(adata_ref, color=elem, size=0.5,
               color_map="Reds", vmax=0.2)


In [ ]:
adataAR = adata_ref[adata_ref[: , 'AR'].X > 0.35, :] 
adataAR

In [ ]:
sc.tl.dendrogram(adataAR, groupby='leiden')

sc.pl.dotplot(adataAR, ["CDKN1A", "CDKN2A", "EPCAM", "CDH1", "ASCL1", "AR"],
              groupby='leiden', dendrogram=True, standard_scale=None )

In [ ]:
adataUnpert = adataALL_new[adataALL_new.obs["sample_name"] == '1',:]

In [ ]:
sc.pl.dotplot(adataUnpert, ["CDKN1A", "CDKN2A", "EPCAM", "CDH1", "ASCL1", "AR"],
              groupby='Sample_type', dendrogram=True , standard_scale='var')

In [ ]:
adataUnpertNEPC = adataUnpert[adataUnpert.obs["Sample_type"] == "NEPC",:]
adataUnpertNEPC[adataUnpertNEPC[: , 'AR'].X > 1, :] 

In [ ]:
adataUnpertAdeno = adataUnpert[adataUnpert.obs["Sample_type"] == "adenocarcinoma",:]
adataUnpertAdeno[adataUnpertAdeno[: , 'AR'].X > 2, :] 

In [ ]:
sc.tl.dendrogram(adataUnpertNEPC, groupby='leiden')

sc.pl.dotplot(adataUnpertNEPC, ["CDKN1A", "CDKN2A", "EPCAM", "CDH1", "ASCL1", "AR"],
              groupby='leiden', dendrogram=True, standard_scale='var' )

In [ ]:
sc.tl.dendrogram(adataUnpertAdeno, groupby='leiden')

sc.pl.dotplot(adataUnpertAdeno, ["CDKN1A", "CDKN2A", "EPCAM", "CDH1", "ASCL1", "AR"],
              groupby='leiden', dendrogram=True, standard_scale='var' )